# SE446 Big Data Engineering Project - Milestone 2

## Chicago Crime Analytics using Apache Spark MLlib

**Course:** SE446 - Big Data Engineering  
**Project:** Chicago Crime Analytics  
**Group:** Group 44  
**Student:** Abdulrahman Meir  
**Repository:** `se446-project-group-44`

This notebook documents Milestone 2 for the Chicago Crimes project. It includes Spark DataFrame analytics, Spark SQL analytics, machine learning preprocessing, Logistic Regression, Random Forest, Gradient-Boosted Tree, CrossValidator tuning, model comparison, feature importance, and cluster execution evidence.


## 1. Environment and Spark Session

The project was executed on the university Spark cluster using YARN.  
The cluster's default PySpark was using Python 3.12 without NumPy, so the Spark jobs were submitted using Python 3.10.

Spark-submit configuration used in terminal:

```bash
--conf spark.pyspark.python=/usr/bin/python3.10
--conf spark.pyspark.driver.python=/usr/bin/python3.10
```


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

spark = (
    SparkSession.builder
    .appName("SE446_M2_Notebook_Group44")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark Version:", spark.version)
print("Spark Master:", spark.sparkContext.master)
print("Application ID:", spark.sparkContext.applicationId)


Spark Version: 3.5.4
Spark Master: yarn
Application ID: application_1771402826595_0330


## 2. Dataset Loading

The dataset is stored on HDFS. The sample file was used for final cluster execution because the university cluster had limited memory during model training.

HDFS paths:

```text
hdfs:///data/chicago_crimes.csv
hdfs:///data/chicago_crimes_sample.csv
```


In [ ]:
input_path = "hdfs:///data/chicago_crimes_sample.csv"

df = spark.read.csv(input_path, header=True, inferSchema=True)

print("Total rows:", df.count())
print("Total columns:", len(df.columns))
df.printSchema()
df.show(5, truncate=False)


Total rows: 10000
Total columns: 22

root
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: integer (nullable = true)
 |-- Y Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)

Displayed first 5 records from the dataset.


## 3. Spark DataFrame Analytics

This section covers basic analytics using Spark DataFrame operations:

1. Crimes per year  
2. Crime type distribution  
3. Arrest distribution  
4. District-level crime count


In [ ]:
print("Crimes Per Year")
df.groupBy("Year").count().orderBy("Year").show(30)

print("Top 10 Crime Types")
df.groupBy("Primary Type").count().orderBy(col("count").desc()).show(10, truncate=False)

print("Arrest Distribution")
df.groupBy("Arrest").count().show()

print("Top 10 Districts by Crime Count")
df.groupBy("District").count().orderBy(col("count").desc()).show(10)


Crimes Per Year:
2001: 4
2002: 2
2003: 1
2004: 6
2005: 19
2006: 4
2007: 7
2008: 16
2009: 5
2010: 5
2011: 7
2012: 9
2013: 10
2014: 16
2015: 28
2016: 20
2017: 49
2018: 28
2019: 36
2020: 25
2021: 83
2022: 135
2023: 9446
2024: 39

Top 10 Crime Types:
THEFT: 2054
BATTERY: 1728
CRIMINAL DAMAGE: 1062
MOTOR VEHICLE THEFT: 948
ASSAULT: 878
DECEPTIVE PRACTICE: 799
OTHER OFFENSE: 586
ROBBERY: 508
BURGLARY: 316
WEAPONS VIOLATION: 284

Arrest Distribution:
true: 1283
false: 8717


### DataFrame Analytics Interpretation

The sample dataset contains 10,000 records and 22 columns. Theft is the most frequent crime type, followed by battery and criminal damage. The arrest distribution is strongly imbalanced because only 1,283 records resulted in arrest while 8,717 did not.


## 4. Spark SQL Analytics

This section uses `spark.sql()` as required by the milestone. It includes:

1. Location hotspots  
2. Overall arrest rate  
3. Arrest rate by crime type  
4. Top crime types by district


In [ ]:
df.createOrReplaceTempView("crimes")

print("Location Hotspots using Spark SQL")
spark.sql("""
    SELECT
        `Location Description`,
        COUNT(*) AS total_crimes
    FROM crimes
    WHERE `Location Description` IS NOT NULL
    GROUP BY `Location Description`
    ORDER BY total_crimes DESC
    LIMIT 10
""").show(10, truncate=False)

print("Overall Arrest Rate using Spark SQL")
spark.sql("""
    SELECT
        COUNT(*) AS total_records,
        SUM(CASE WHEN Arrest = true THEN 1 ELSE 0 END) AS arrest_count,
        SUM(CASE WHEN Arrest = false THEN 1 ELSE 0 END) AS non_arrest_count,
        ROUND(
            SUM(CASE WHEN Arrest = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) AS arrest_rate_percent
    FROM crimes
    WHERE Arrest IS NOT NULL
""").show(truncate=False)

print("Arrest Rate by Crime Type using Spark SQL")
spark.sql("""
    SELECT
        `Primary Type`,
        COUNT(*) AS total_records,
        SUM(CASE WHEN Arrest = true THEN 1 ELSE 0 END) AS arrest_count,
        SUM(CASE WHEN Arrest = false THEN 1 ELSE 0 END) AS non_arrest_count,
        ROUND(
            SUM(CASE WHEN Arrest = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) AS arrest_rate_percent
    FROM crimes
    WHERE Arrest IS NOT NULL
    GROUP BY `Primary Type`
    ORDER BY arrest_rate_percent DESC
    LIMIT 10
""").show(10, truncate=False)


Location Hotspots:
STREET: 2737
APARTMENT: 1909
RESIDENCE: 1358
SIDEWALK: 536
PARKING LOT / GARAGE (NON RESIDENTIAL): 362
SMALL RETAIL STORE: 234
ALLEY: 219
RESTAURANT: 187
OTHER (SPECIFY): 156
VEHICLE NON-COMMERCIAL: 139

Overall Arrest Rate:
total_records: 10000
arrest_count: 1283
non_arrest_count: 8717
arrest_rate_percent: 12.83

Highest Arrest Rate by Crime Type:
GAMBLING: 100.00%
LIQUOR LAW VIOLATION: 100.00%
OBSCENITY: 100.00%
NARCOTICS: 99.37%
CONCEALED CARRY LICENSE VIOLATION: 83.33%
INTERFERENCE WITH PUBLIC OFFICER: 80.77%
PUBLIC PEACE VIOLATION: 58.82%
WEAPONS VIOLATION: 52.46%
HUMAN TRAFFICKING: 50.00%
HOMICIDE: 50.00%


### Spark SQL Interpretation

Spark SQL shows that the most common crime locations are streets, apartments, and residences. The overall arrest rate is 12.83%, confirming a strong class imbalance. Arrest rate varies greatly by crime type, which supports using `Primary Type` as an important feature in the ML pipeline.


## 5. Data Cleaning and Feature Engineering

The target label is `Arrest`.

The selected ML features are:

- `Primary Type`
- `District`
- `Year`
- `Domestic`

Preprocessing steps:

- Remove records where `Arrest` is null.
- Fill missing categorical and numeric values.
- Convert `Arrest` to numeric label.
- Convert `Domestic` to numeric feature.
- Encode `Primary Type` using `StringIndexer`.
- Assemble all features into a vector using `VectorAssembler`.


In [ ]:
df_ml = df.select("Primary Type", "District", "Year", "Domestic", "Arrest")
df_ml = df_ml.filter(col("Arrest").isNotNull())

df_ml = df_ml.fillna({
    "Primary Type": "UNKNOWN",
    "District": -1,
    "Year": -1,
    "Domestic": False
})

df_ml = (
    df_ml
    .withColumn("label", col("Arrest").cast("int"))
    .withColumn("DomesticIndex", col("Domestic").cast("int"))
)

df_ml.groupBy("label").count().show()

primary_type_indexer = StringIndexer(
    inputCol="Primary Type",
    outputCol="PrimaryTypeIndex",
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=["PrimaryTypeIndex", "District", "Year", "DomesticIndex"],
    outputCol="features",
    handleInvalid="keep"
)

train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_data.count())
print("Testing rows:", test_data.count())


Label Distribution:
label 0: 8717
label 1: 1283

Training rows: 8079
Testing rows: 1921


## 6. Model Evaluation Setup

The project evaluates models using:

- Accuracy
- F1 Score
- Weighted Precision
- Weighted Recall
- AUC
- Confusion Matrix

Because the dataset is imbalanced, F1 score and confusion matrix are especially important.


In [ ]:
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)


## 7. Logistic Regression

Logistic Regression was used as the baseline classification model.


In [ ]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=10,
    regParam=0.1
)

lr_pipeline = Pipeline(stages=[
    primary_type_indexer,
    assembler,
    lr
])

lr_model = lr_pipeline.fit(train_data)
lr_predictions = lr_model.transform(test_data)

print("Accuracy:", accuracy_evaluator.evaluate(lr_predictions))
print("F1 Score:", f1_evaluator.evaluate(lr_predictions))
print("Precision:", precision_evaluator.evaluate(lr_predictions))
print("Recall:", recall_evaluator.evaluate(lr_predictions))
print("AUC:", auc_evaluator.evaluate(lr_predictions))

lr_predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()


MODEL RESULTS: Logistic Regression
Accuracy: 0.8787
F1 Score: 0.8220
Precision: 0.7721
Recall: 0.8787
AUC: 0.6716

Confusion Matrix:
label=0, prediction=0.0: 1688
label=1, prediction=0.0: 233


### Logistic Regression Interpretation

Logistic Regression achieved high accuracy, but it predicted all test records as non-arrest. This shows the effect of class imbalance. Accuracy alone is misleading, so F1 score, AUC, and confusion matrix are needed.


## 8. Random Forest

Random Forest was used as a tree-based ensemble model. The number of trees and tree depth were reduced to avoid memory termination on the university cluster.


In [ ]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=3,
    maxDepth=3,
    seed=42
)

rf_pipeline = Pipeline(stages=[
    primary_type_indexer,
    assembler,
    rf
])

rf_model = rf_pipeline.fit(train_data)
rf_predictions = rf_model.transform(test_data)

print("Accuracy:", accuracy_evaluator.evaluate(rf_predictions))
print("F1 Score:", f1_evaluator.evaluate(rf_predictions))
print("Precision:", precision_evaluator.evaluate(rf_predictions))
print("Recall:", recall_evaluator.evaluate(rf_predictions))
print("AUC:", auc_evaluator.evaluate(rf_predictions))

rf_predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

rf_classifier = rf_model.stages[-1]
feature_columns = ["PrimaryTypeIndex", "District", "Year", "DomesticIndex"]
for feature, importance in sorted(zip(feature_columns, rf_classifier.featureImportances.toArray()), key=lambda x: x[1], reverse=True):
    print(feature, importance)


MODEL RESULTS: Random Forest
Accuracy: 0.8787
F1 Score: 0.8220
Precision: 0.7721
Recall: 0.8787
AUC: 0.5913

Confusion Matrix:
label=0, prediction=0.0: 1688
label=1, prediction=0.0: 233

Feature Importance:
PrimaryTypeIndex: 0.4423
District: 0.3991
Year: 0.1411
DomesticIndex: 0.0175


### Random Forest Interpretation

Random Forest also predicted most records as non-arrest, but it provided useful feature importance. The most important features were `PrimaryTypeIndex` and `District`.


## 9. Gradient-Boosted Tree

Gradient-Boosted Tree was used as another ensemble model. It performed better than Logistic Regression and Random Forest on arrest detection.


In [ ]:
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=3,
    maxDepth=2,
    seed=42
)

gbt_pipeline = Pipeline(stages=[
    primary_type_indexer,
    assembler,
    gbt
])

gbt_model = gbt_pipeline.fit(train_data)
gbt_predictions = gbt_model.transform(test_data)

print("Accuracy:", accuracy_evaluator.evaluate(gbt_predictions))
print("F1 Score:", f1_evaluator.evaluate(gbt_predictions))
print("Precision:", precision_evaluator.evaluate(gbt_predictions))
print("Recall:", recall_evaluator.evaluate(gbt_predictions))
print("AUC:", auc_evaluator.evaluate(gbt_predictions))

gbt_predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

gbt_classifier = gbt_model.stages[-1]
feature_columns = ["PrimaryTypeIndex", "District", "Year", "DomesticIndex"]
for feature, importance in sorted(zip(feature_columns, gbt_classifier.featureImportances.toArray()), key=lambda x: x[1], reverse=True):
    print(feature, importance)


MODEL RESULTS: Gradient-Boosted Tree
Accuracy: 0.8938
F1 Score: 0.8588
Precision: 0.8915
Recall: 0.8938
AUC: 0.7650

Confusion Matrix:
label=0, prediction=0.0: 1683
label=0, prediction=1.0: 5
label=1, prediction=0.0: 199
label=1, prediction=1.0: 34

Feature Importance:
PrimaryTypeIndex: 0.9780
District: 0.0220
Year: 0.0000
DomesticIndex: 0.0000


### Gradient-Boosted Tree Interpretation

Gradient-Boosted Tree performed better than Logistic Regression and Random Forest. It predicted some arrest cases correctly and achieved the strongest AUC among the non-tuned models. The most important feature was `PrimaryTypeIndex`.


## 10. CrossValidator Hyperparameter Tuning

CrossValidator was used with a Random Forest classifier. The parameter grid was intentionally kept small because the university cluster had limited memory.


In [ ]:
rf_tuning = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    seed=42
)

tuning_pipeline = Pipeline(stages=[
    primary_type_indexer,
    assembler,
    rf_tuning
])

param_grid = (
    ParamGridBuilder()
    .addGrid(rf_tuning.numTrees, [2, 3])
    .addGrid(rf_tuning.maxDepth, [2, 3])
    .build()
)

cross_validator = CrossValidator(
    estimator=tuning_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=f1_evaluator,
    numFolds=2,
    seed=42,
    parallelism=1
)

cv_model = cross_validator.fit(train_data)
cv_predictions = cv_model.transform(test_data)

print("Accuracy:", accuracy_evaluator.evaluate(cv_predictions))
print("F1 Score:", f1_evaluator.evaluate(cv_predictions))

cv_predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

best_rf_model = cv_model.bestModel.stages[-1]
print("Best numTrees:", best_rf_model.getNumTrees)
print("Best maxDepth:", best_rf_model.getOrDefault("maxDepth"))


MODEL RESULTS: Tuned Random Forest with CrossValidator
Accuracy: 0.8995
F1 Score: 0.8767

Confusion Matrix:
label=0, prediction=0.0: 1669
label=0, prediction=1.0: 19
label=1, prediction=0.0: 174
label=1, prediction=1.0: 59

Best Parameters:
numTrees: 2
maxDepth: 2

Feature Importance:
PrimaryTypeIndex: 0.9655
DomesticIndex: 0.0345
District: 0.0000
Year: 0.0000


### CrossValidator Interpretation

CrossValidator improved the F1 score compared with the untuned Random Forest. It also predicted more arrest cases correctly. The grid was small due to cluster limitations, but it still demonstrates Spark MLlib hyperparameter tuning with `ParamGridBuilder` and `CrossValidator`.


## 11. Model Comparison

| Model | Accuracy | F1 Score | Precision | Recall | AUC |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.8787 | 0.8220 | 0.7721 | 0.8787 | 0.6716 |
| Random Forest | 0.8787 | 0.8220 | 0.7721 | 0.8787 | 0.5913 |
| Gradient-Boosted Tree | 0.8938 | 0.8588 | 0.8915 | 0.8938 | 0.7650 |
| Tuned Random Forest with CrossValidator | 0.8995 | 0.8767 | N/A | N/A | N/A |

The best model based on F1 score was the tuned Random Forest with CrossValidator. The Gradient-Boosted Tree achieved the strongest AUC among the non-tuned models.


## 12. Spark-Submit Cluster Execution Evidence

All tasks were executed using `spark-submit` on YARN.

Example command:

```bash
spark-submit \
  --master yarn \
  --deploy-mode client \
  --conf spark.pyspark.python=/usr/bin/python3.10 \
  --conf spark.pyspark.driver.python=/usr/bin/python3.10 \
  --conf spark.ui.showConsoleProgress=false \
  m2_task1_analytics.py 2>&1 | tee output/milestone2/task1_analytics_output.txt
```

Output logs are stored in:

```text
output/milestone2/
```

Files:

```text
task1_analytics_output.txt
task2_logistic_output.txt
task3_random_forest_output.txt
task4_gbt_output.txt
task5_crossvalidator_output.txt
```


## 13. Challenges and Solutions

### Challenge 1: Python 3.12 Missing NumPy

The default PySpark environment used Python 3.12, but Spark MLlib required NumPy. Python 3.10 had NumPy installed, so the jobs were submitted using Python 3.10.

### Challenge 2: Cluster Memory Limits

The original full pipeline was implemented in `m2_full_pipeline_attempt.py`, but it was too heavy for the cluster and was terminated during training. The solution was to split the work into independent Spark-submit scripts.

### Challenge 3: Class Imbalance

The dataset is imbalanced, with 8,717 non-arrest records and 1,283 arrest records. Multiple metrics were used to avoid relying on accuracy alone.


## 14. Conclusion

Milestone 2 successfully demonstrates a complete Spark MLlib workflow:

```text
HDFS → Spark DataFrames → Spark SQL Analytics → Feature Engineering → Spark MLlib Models → Evaluation → Interpretation
```

The project loaded the Chicago Crimes dataset from HDFS, performed Spark DataFrame and Spark SQL analytics, trained multiple ML models, evaluated model performance, used CrossValidator for tuning, and saved full terminal output logs from YARN execution.


In [ ]:
spark.stop()
print("Notebook workflow completed.")


Notebook workflow completed.
